# Letting the LLM read the SysML

This project used to hand-parse SysML v2. `sysml/pipeline/parse.py` was 791 lines of
grammar -- a scanner, a declaration splitter, a reference resolver -- and it produced
`out/model.json`, which two more steps turned into a graph. That is gone. The graph is
now built by **graphrag_importer's own extraction pipeline**, the same code the
platform's importer pods run, called in-process against a local ArangoDB.

This notebook is about what changed and what it bought. `simple-demo.ipynb` is about
asking the resulting graph questions, and it needed almost no changes -- which is
itself part of the point.

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
export CHAT_API_KEY=sk-...
python build.py
```

## The whole build

Two steps. `extract` runs the LLM over the source text and leaves a NetworkX graph
and some JSON in `out/kg`; `load` hands those files to the importer's own ArangoDB
writer. Neither knows anything about SysML.

In [1]:
import inspect, logging
from sysml import config, nl
from sysml.pipeline import analogy, extract, load

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

print(inspect.getsource(extract.graphrag))

def graphrag():
    """The extraction pipeline, configured and pointed at out/kg.

    `enable_chunk_embeddings` is on because the `unified` retriever searches the
    source chunks in parallel with the entity graph, and without vectors on the
    chunks that half of it has nothing to search.

    The LLM cache lives in the working directory, so a second run over unchanged
    files re-reads the answers instead of re-buying them.
    """
    config.openai_key()
    config.KG.mkdir(parents=True, exist_ok=True)
    # corpus_graph and the builder both attach a stdout handler at import time,
    # which corrupts anything reading stdout. Keep the logs, move them to stderr.
    for name in ("arango-graphrag", "vectordb", "graphrag"):
        log = logging.getLogger(name)
        log.handlers = [logging.StreamHandler(sys.stderr)]
        log.propagate = False

    from graphrag.graph_builder.builder.graphrag import GraphRAG

    return GraphRAG(
        working_dir=str(config.KG),
        ent

In [2]:
print(inspect.getsource(load.load))

async def load() -> dict:
    db = config.db(create=True)
    reset(db)

    imp = importer()
    await imp.initialize(config.token())
    await imp.import_documents(config.ARTIFACTS.FULL_DOCS)
    # Deliberately without the chunk-embedding file -- see `chunk_vectors`.
    await imp.import_text_chunks(config.ARTIFACTS.TEXT_CHUNKS)
    chunk_vectors(db, imp)
    await imp.import_entities(config.ARTIFACTS.ENTITIES)
    await imp.import_relationships(config.ARTIFACTS.RELATIONSHIPS)
    await imp.import_community_reports(config.ARTIFACTS.COMMUNITY_REPORTS)

    # Not the edge collection. ArangoDB requires the indexed vector field on every
    # document in the collection, and only RELATED_TO edges carry one -- the
    # structural edges have no text to embed. Exact COSINE_SIMILARITY is used for
    # those instead, which is also the only form that can be filtered by
    # relationship_type in the same query.
    for name in (config.ENTITIES, config.CHUNKS, config.COMMUNITIES):
        awai

That is the whole of it. `GraphRAG` and `ImportGraphToADB` are imported unmodified;
the only local code is the two `await` sequences above, plus a status-sink shim
because the writer announces its progress to a platform service that is not here.

## The ontology is the only thing we tell it about SysML

`entity_types` and `relationship_types` are constructor arguments. With
`enable_strict_types=True` they are a closed vocabulary rather than a hint: an
entity or edge whose type is not on the list is **dropped**, not renamed.

These two lists are exactly what the parser used to recognise in its grammar. The
same 27 declaration kinds and the same 18 relations -- moved out of Python and into
the prompt.

In [3]:
print(f"{len(config.KINDS)} entity types")
print("  " + ", ".join(config.KINDS))
print(f"\n{len(config.RELATIONS_ONTOLOGY)} relation types")
print("  " + ", ".join(config.RELATIONS_ONTOLOGY))

27 entity types
  Package, Part, Action, State, Port, Item, Attribute, Requirement, Calc, Analysis, Connection, Interface, View, Viewpoint, Enumeration, Concern, Constraint, Flow, Allocation, Event, Metadata, UseCase, Rendering, Verification, Snapshot, Timeslice, Occurrence

18 relation types
  owns, typedBy, specializes, redefines, satisfies, refines, derives, performs, subject, exhibits, connects, transitionsTo, variantOf, imports, sliceOf, sends, dependsOn, valueRef


## What came out

The database is built by `python build.py`. This connects to it.

In [4]:
db = config.db()
for name in config.ALL_COLLECTIONS:
    print(f"{db.collection(name).count():>6}  {name}")

    30  sysml_Documents
   114  sysml_Chunks
  2503  sysml_Entities
   138  sysml_Communities
  6962  sysml_Relations


## 1. The ontology held

Every relation type the extraction produced, counted. All of them are from the list
above -- `enable_strict_types` guarantees it, and the last cell checks rather than
assumes.

The shape is different from the parser's, and honestly so. The parser emitted an
`owns` edge for every containment in the file, so `owns` dominated everything;
extraction reports containment only where the text makes a point of it, and
`refines` -- a relationship between two *statements*, which is what a requirements
model is mostly made of -- comes out on top instead.

In [5]:
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT kind = r.relationship_type WITH COUNT INTO n
  SORT n DESC RETURN {{kind, n}}'''
rows = list(db.aql.execute(Q))
for row in rows:
    print(f"{row['n']:>6}  {row['kind']}")

off = [r["kind"] for r in rows if r["kind"] not in config.RELATIONSHIP_TYPES]
print(f"\n{len(rows)} of {len(config.RELATIONS_ONTOLOGY)} offered types used, "
      f"{len(off)} off-ontology: {off or 'none'}")

   354  refines
   260  satisfies
    86  typedby
    85  imports
    68  dependson
    68  owns
    64  performs
    31  connects
    23  exhibits
    23  transitionsto
    16  sliceof
    11  specializes
    11  variantof
     8  derives
     8  subject
     2  valueref
     1  redefines

17 of 18 offered types used, 0 off-ontology: none


## 2. It reads the prose, not just the syntax

`dependsOn` is in our ontology but there is no SysML keyword for it. The parser
could only ever emit a relation that some statement spelled out. These edges exist
because the text explains a dependency in words.

In [6]:
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.relationship_type == "dependson"
  LIMIT 4
  RETURN {{a: DOCUMENT(r._from).entity_name, b: DOCUMENT(r._to).entity_name,
          why: r.description}}'''
for row in db.aql.execute(Q):
    print(f"{row['a']}  ->  {row['b']}")
    print(f"    {row['why']}\n")

COSMAQUANTITIESANDUNITSPACKAGE  ->  DATAVALUE
    CoSMAQuantitiesAndUnitsPackage depends on DataValue from the Base package for logarithmic calculations and potential data values.

CALCULATE DELTA V  ->  CALCULATE LAUNCH STAGE DELTA V
    Calculate Launch Stage Delta V relies on Calculate Delta V to compute the delta-v using specific initial and final masses and specific impulse.

CALCULATE DELTA V  ->  CALCULATE SPACECRAFT BURN DELTA V
    Calculate Spacecraft Burn Delta V uses Calculate Delta V to compute delta-v using the spacecraft's specific impulse, initial mass, and final mass.

CALCULATE LAUNCH STAGE DELTA V  ->  SUM GROSS MASS
    Calculate Launch Stage Delta V uses Sum Gross Mass to determine the total mass of payload components for initial mass calculation.



## 3. The same element in two files is one element

The parser keyed elements on their qualified name inside one file, so a requirement
declared in one package and referred to from another became two unrelated rows.
Extraction merges by name across the whole corpus, so an element accumulates every
file it appears in -- and the answer to "where is this discussed" stops being "one
place".

In [7]:
# Packages are excluded: the standard-library ones are imported by everything, so
# they crowd out the elements the question is actually about.
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER LENGTH(e.files) > 1 AND e.entity_type != "package"
  SORT LENGTH(e.files) DESC LIMIT 6
  RETURN {{name: e.entity_name, type: e.entity_type, files: LENGTH(e.files)}}'''
total = next(iter(db.aql.execute(
    f'''RETURN LENGTH(FOR e IN {config.ENTITIES}
         FILTER LENGTH(e.files) > 1 AND e.entity_type != "package" RETURN 1)''')))
print(f"{total} non-package elements appear in more than one source file. The widest:")
for row in db.aql.execute(Q):
    print(f"  {row['files']:>3} files   {row['name']} ({row['type']})")

599 non-package elements appear in more than one source file. The widest:
    7 files   SPACECRAFT (part)
    5 files   SYSTEM (part)
    5 files   PLANETARYPROTECTION (part)
    5 files   APOLLO11MISSION (part)
    4 files   INSTRUMENTUNIT (part)
    4 files   COMMAND MODULE (part)


The same merge across model boundaries is more interesting. Nothing was told that
the two drone files describe one vehicle -- the names simply agreed.

In [8]:
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER LENGTH(e.models) > 1
  RETURN {{name: e.entity_name, type: e.entity_type, models: e.models}}'''
for row in db.aql.execute(Q):
    print(f"  {row['name']:<16} {row['type']:<12} {', '.join(row['models'])}")

  CONTROL          requirement  apollo-11, drone-logical
  SCALARVALUES     package      apollo-11, drone-logical
  STATUS           attribute    apollo-11, drone-logical
  DRONE            part         drone-base, drone-logical
  BATTERY          part         drone-base, drone-logical


## 4. Communities and their reports come with it

The old `enrich` step hand-rolled label propagation over the traceability edges,
picked a title from the largest member, and wrote its own report prompt -- about 200
lines. Leiden and the report writer are part of the extraction pipeline, so all of
that is now upstream's. Three levels of hierarchy, and the reports are structured
rather than a blob of text.

In [9]:
Q = f'''
FOR c IN {config.COMMUNITIES}
  FILTER c.level == 0
  SORT c.occurrence DESC LIMIT 1
  RETURN c'''
c = next(iter(db.aql.execute(Q)))
print(f"level {c['level']}   covers {c['occurrence']:.0%} of the corpus   "
      f"{len(c['sub_communities'])} sub-communities\n")
print(c["report_json"]["title"])
print(c["report_json"]["summary"][:400], "...\n")
for f in c["report_json"]["findings"][:2]:
    print(f"- {f['summary'] if isinstance(f, dict) else f}")

level 0   covers 100% of the corpus   4 sub-communities

Apollo11Model Software Package Community
The community is centered around the Apollo11Model software package, which plays a crucial role in integrating various components and packages associated with the Apollo 11 mission. Key packages such as the MissionPackage, CalculationsPackage, and CoSMAPackage are instrumental in providing functionalities, operational details, and modeling capabilities to support mission simulation and execution. ...

- Central Role of Apollo11Model
- MissionPackage Integration


## What it costs

Nothing here is free, and three of these are real losses.

| | parser | extraction |
|---|---|---|
| element names | `SaturnV`, as declared | `SATURNV` -- upper-cased at `_op.py:264` |
| where it is declared | `file:line` | the files it appears in; no line |
| `dryMass = 137000 [kg]` | a typed field you can sum in AQL | a number inside a sentence |
| rebuild | free and identical | cached, but non-deterministic on a change |

The line numbers and the structured attribute values are the ones that hurt: any
question that was arithmetic over `attributes` is now a question about prose. What
replaces them is that the numbers are still *there*, in the description, where a
retrieval answer can quote them -- just not where an AQL `SUM()` can reach them.

In [10]:
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER CONTAINS(LOWER(e.description), "dry mass")
  LIMIT 2 RETURN {{name: e.entity_name, files: e.files, d: e.description}}'''
for row in db.aql.execute(Q):
    print(f"{row['name']}   {row['files'][0]}")
    print(f"    {row['d'][:200]}\n")

CALCULATE SPACECRAFT BURN DELTA V   apollo-11-sysml-v2/Analysis/CalculationsPackage.sysml
    Calculates the delta-v provided by a self-propelled spacecraft, using inputs including the spacecraft's dry mass and propellant mass.

SUM DRY MASS   apollo-11-sysml-v2/Analysis/CalculationsPackage.sysml
    A utility to sum the dry mass of a set of components, specifically FuelledComponent.



## The size of the change

The parser, the projection and the enrichment were 1,557 lines between them, all of
it ours to maintain and all of it specific to one input language. What replaced them
is a fifth of that, and a good part of the remainder is the comments explaining the
upstream bugs the local path runs into.

In [11]:
from pathlib import Path
import subprocess

old = {"parse.py": 791, "project.py": 298, "enrich.py": 468}
new = {p.name: len(p.read_text(encoding="utf-8").splitlines())
       for p in [Path("sysml/pipeline/extract.py"), Path("sysml/pipeline/load.py")]}
print(f"removed  {sum(old.values()):>5}   " + ", ".join(f"{k} {v}" for k, v in old.items()))
print(f"added    {sum(new.values()):>5}   " + ", ".join(f"{k} {v}" for k, v in new.items()))
print(f"net      {sum(new.values()) - sum(old.values()):>5} lines")

removed   1557   parse.py 791, project.py 298, enrich.py 468
added      326   extract.py 119, load.py 207
net      -1231 lines


## What did not change

`sysml/nl.py`, the entire read side, kept working. The retrievers and the AQLizer
were pointed at a graph the importer's own writer produced, which is what they were
built to read -- where before they were pointed at a hand-made imitation of it.
`aql_examples.md` had to be rewritten, because the fields it teaches are different
ones, but no Python moved.

That is the argument for the change in one line: the graph is now produced by the
code that owns the schema, so agreeing with the schema is no longer something this
project has to keep doing by hand.

In [12]:
print((await nl.retriever().ask_async(
    "What is the drone battery for and what capacity does it have?")).answer[:700])

## Function of the Drone Battery

The drone battery in the model is a component within the `Drone_SystemArchitecture` designed to provide energy storage capability for the drone. It connects to a power management module which is responsible for monitoring and controlling the battery's function, including its charging and discharging processes. The battery system includes various parts such as battery cells, a battery management system, a protection circuit, and a power connector, among others, to ensure proper and efficient operation within the drone's power system [CITE:1][CITE:3].

## Capacity of the Drone Battery

The specified capacity of the drone battery within the `Drone_SystemArchite
